<a href="https://colab.research.google.com/github/AnzorGozalishvili/IOAI-2025-lectures/blob/main/lecture_31_transformers_finetuning_on_classification/notebooks/Davit6174_georgian_distilbert_mlm_%26_DGurgurov_georgian_sa_Sentiment_Analysis_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Davit6174/georgian-distilbert-mlm
- Model Card: https://huggingface.co/Davit6174/georgian-distilbert-mlm
- Dataset Source (for tokenizer test): https://huggingface.co/datasets/DGurgurov/georgian_sa/viewer/default/test?views%5B%5D=test

In [1]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
from transformers import pipeline

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("Davit6174/georgian-distilbert-mlm")
# Load the PyTorch model instead of the TensorFlow model
model = AutoModelForMaskedLM.from_pretrained("Davit6174/georgian-distilbert-mlm", from_tf=True)

# Build pipeline
mask_filler = pipeline(
    "fill-mask", model=model, tokenizer=tokenizer
)

text = 'ქართული <mask> [MASK] სწავლა საკმაოდ რთულია'

# Generate model output
preds = mask_filler(text)

# Print top 5 predictions
for pred in preds:
    print(f">>> {pred['sequence']}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
All TF 2.0 model weights were used when initializing DistilBertForMaskedLM.

Some weights of DistilBertForMaskedLM were not initialized from the TF 2.0 model and are newly initialized: ['vocab_projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


>>> ქართული < mask > ქართული სწავლა საკმაოდ რთულია
>>> ქართული < mask > - სწავლა საკმაოდ რთულია
>>> ქართული < mask > > სწავლა საკმაოდ რთულია
>>> ქართული < mask >, სწავლა საკმაოდ რთულია
>>> ქართული < mask > სწავლა სწავლა საკმაოდ რთულია


# Prepare Model For Sentiment Classification

In [2]:
from transformers import AutoModelForSequenceClassification

# Load the sequence classification model
num_labels = 2 # Binary classification (positive/negative)
model = AutoModelForSequenceClassification.from_pretrained("Davit6174/georgian-distilbert-mlm", num_labels=num_labels, from_tf=True)


All TF 2.0 model weights were used when initializing DistilBertForSequenceClassification.

All the weights of DistilBertForSequenceClassification were initialized from the TF 2.0 model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use DistilBertForSequenceClassification for predictions without further training.


# Load Sentiment Dataset

In [3]:
!pip install datasets

In [4]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("DGurgurov/georgian_sa")
print(dataset)
pd.concat([pd.Series(dataset[split]['label']).value_counts().T for split in ['train', 'validation', 'test']], axis=1, keys=['train', 'validation', 'test'])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1080
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 120
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 330
    })
})


,train,validation,test
0,550,50,165
1,530,70,165


# Train

In [5]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score

# Define training arguments
training_args = TrainingArguments(
    # output_dir="./sentiment_classification_results",  # Output directory
    output_dir=None,
    evaluation_strategy="epoch",  # Evaluate at the end of each epoch
    learning_rate=2e-5,  # Learning rate
    per_device_train_batch_size=16,  # Batch size per device during training
    per_device_eval_batch_size=16,  # Batch size for evaluation
    num_train_epochs=3,  # Number of training epochs
    weight_decay=0.01,  # Weight decay
    push_to_hub=False,  # Whether to upload the model to the Hugging Face Model Hub
    # logging_dir='./logs',
    logging_dir=None,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    # Align save_strategy with evaluation_strategy
    save_strategy="epoch",
    report_to="none",
)

# Function to tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Tokenize the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Function to compute metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

# Create Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],  # Use the tokenized training dataset
    eval_dataset=tokenized_datasets["validation"],  # Use the tokenized validation dataset
    tokenizer=tokenizer,
    compute_metrics=compute_metrics, # Pass the compute_metrics function

)

# Train the model
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Map:   0%|          | 0/120 [00:00<?, ? examples/s]

<ipython-input-5-974f810e3828>:40: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.533400,0.468623,0.808333
2,0.441400,0.390484,0.816667
3,0.331700,0.347623,0.866667


TrainOutput(global_step=204, training_loss=0.4627510928640179, metrics={'train_runtime': 188.2533, 'train_samples_per_second': 17.211, 'train_steps_per_second': 1.084, 'total_flos': 429194371645440.0, 'train_loss': 0.4627510928640179, 'epoch': 3.0})

# Explain Model Predictions

In [29]:
!pip install transformers-interpret

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.4 MB/s eta 0:00:00


In [31]:
from transformers_interpret import SequenceClassificationExplainer

cls_explainer = SequenceClassificationExplainer(model, tokenizer)

In [55]:
def interpret_text(text):
  word_attributions = cls_explainer(text)
  return cls_explainer.visualize()


for text, sentiment in zip(dataset['test']['text'][:10], dataset['test']['label'][:10]):
    interpret_text(text)
    print(sentiment)

True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
0,LABEL_0 (0.73),LABEL_0,1.72,"[CLS] ყოფილი პრემ ##ეირ ##ი ხო ##კრ ##იშ ##ვილზე საუბრობს და აცხადებს , რომ მის ##თი ##ვს ის ყველაზე სიმპ ##ატი ##ური მინისტრი იყო [SEP]"


1


1


1


True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
1,LABEL_1 (0.94),LABEL_1,1.56,[CLS] მან მადლობა გადაუხადა საქართველოს ნატოს ავღანეთის მისია ##ში შეტანილი წვლილი ##სთ ##ვის [SEP]


...

True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
0,LABEL_0 (0.85),LABEL_0,2.66,"[CLS] რაც შეეხება გია გაჩეჩილ ##აძის მხარეს , ისინი თბილისის საქალაქო სასამართლოს მოსამართ ##ლოს გადაწყვეტილებით კმაყოფილ ##ები არიან [SEP]"


1


True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
1,LABEL_1 (0.93),LABEL_1,1.55,[CLS] ყველას დიდი მადლობა [SEP]


1


1
